# 可选实验 - 简单神经网络
在本实验中，我们将用 NumPy 构建一个小型神经网络。它将与你在 Tensorflow 中实现的“咖啡烘焙”网络相同。
   <center> <img  src="./图片/C2_W1_CoffeeRoasting.png" width="400" />

> **图中文字（中文）**：咖啡烘焙：时长（分钟）与温度（摄氏度）的二维散点图 标注：z ≥ 0.5 ？   <center/>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
import tensorflow as tf
from lab_utils_common import dlc, sigmoid
from lab_coffee_utils import load_coffee_data, plt_roast, plt_prob, plt_layer, plt_network, plt_output_unit
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

## 数据集
这与上一个实验的数据集相同。

In [ ]:
X,Y = load_coffee_data();
print(X.shape, Y.shape)

让我们把下面的咖啡烘焙数据画出来。两个特征是温度（摄氏度）和时长（分钟）。[Coffee Roasting at Home](https://www.merchantsofgreencoffee.com/how-to-roast-green-coffee-in-your-oven/) 建议时长最好保持在 12 到 15 分钟之间，温度应在 175 到 260 摄氏度之间。当然，随着温度升高，时长应缩短。


In [ ]:
plt_roast(X,Y)

### 归一化数据
为了与上一个实验保持一致，我们将对数据做归一化。更多细节请参考那个实验。

In [ ]:
print(f"Temperature Max, Min pre normalization: {np.max(X[:,0]):0.2f}, {np.min(X[:,0]):0.2f}")
print(f"Duration    Max, Min pre normalization: {np.max(X[:,1]):0.2f}, {np.min(X[:,1]):0.2f}")
norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)  # 学习均值、方差
Xn = norm_l(X)
print(f"Temperature Max, Min post normalization: {np.max(Xn[:,0]):0.2f}, {np.min(Xn[:,0]):0.2f}")
print(f"Duration    Max, Min post normalization: {np.max(Xn[:,1]):0.2f}, {np.min(Xn[:,1]):0.2f}")

## NumPy 模型（NumPy 中的前向传播）
<center> <img  src="./图片/C2_W1_RoastingNetwork.PNG" width="200" />

> **图中文字（中文）**：网络结构示意（第 1 层、第 2 层）   <center/>  
让我们构建课上描述的“咖啡烘焙网络”。它有两层，均使用 sigmoid 激活函数。

正如课上所述，你可以用 NumPy 构建自己的密集层。这样就可以用它来构建多层神经网络。

<img src="图片/C2_W1_dense2.PNG" width="600" height="450">

> **图中文字（中文）**：神经网络层 def my_dense(a_in, W, b, g):     units = W.shape[1]     a_out = np.zeros(units)     for j in range(units):         z = np.dot(W[:, j], a_in) + b[j]         a_out[j] = g(z)     return a_out

在第一个可选实验中，你分别用 NumPy 和 Tensorflow 构建了一个神经元，并注意到它们的相似性。一层不过是包含多个神经元/单元。正如课上所述，可以用一个 for 循环访问层中的每个单元（`j`），把该单元的权重（`W[:,j]`）与输入做点积，再加上该单元的偏置（`b[j]`），得到 `z`。然后可以对结果应用激活函数 `g(z)`。下面我们就用这个办法来构建一个“密集层”子程序。

In [ ]:
def my_dense(a_in, W, b, g):
    """计算全连接层
    参数：
      a_in (ndarray (n, )) : 数据，1 个样本
      W    (ndarray (n,j)) : 权重矩阵，每个单元 n 个特征，共 j 个单元
      b    (ndarray (j, )) : 偏置向量，j 个单元
      g    激活函数（例如 sigmoid、relu……）
    返回
      a_out (ndarray (j,))  : j 个单元|"""
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):               
        w = W[:,j]                                    
        z = np.dot(w, a_in) + b[j]         
        a_out[j] = g(z)               
    return(a_out)

下面的单元格利用上面的 `my_dense` 子程序构建了一个两层神经网络。

In [ ]:
def my_sequential(x, W1, b1, W2, b2):
    a1 = my_dense(x,  W1, b1, sigmoid)
    a2 = my_dense(a1, W2, b2, sigmoid)
    return(a2)

我们可以从上一个 Tensorflow 实验中复制已训练好的权重和偏置。

In [ ]:
W1_tmp = np.array( [[-8.93,  0.29, 12.9 ], [-0.1,  -7.32, 10.81]] )
b1_tmp = np.array( [-9.82, -9.28,  0.96] )
W2_tmp = np.array( [[-31.18], [-27.59], [-32.56]] )
b2_tmp = np.array( [15.41] )

### 预测
<img align="left" src="./图片/C2_W1_RoastingDecision.PNG"     style=" width:380px; padding: 10px 20px; " >

> **图中文字（中文）**：标注：z ≥ 0.5 ？

训练好模型后，你就可以用它做预测了。回忆一下，我们模型的输出是一个概率——在这里是“好烘焙”的概率。要做决策，必须把概率与某个阈值比较。这里我们使用 0.5

让我们先写一个类似 Tensorflow `model.predict()` 的程序。它接收一个矩阵 $X$（全部 $m$ 个样本位于各行），运行模型做出预测。

In [ ]:
def my_predict(X, W1, b1, W2, b2):
    m = X.shape[0]
    p = np.zeros((m,1))
    for i in range(m):
        p[i,0] = my_sequential(X[i], W1, b1, W2, b2)
    return(p)

我们可以在两个样本上试试这个程序：

In [ ]:
X_tst = np.array([
    [200,13.9],  # 正例
    [200,17]])   # 负例
X_tstn = norm_l(X_tst)  # 记得做归一化
predictions = my_predict(X_tstn, W1_tmp, b1_tmp, W2_tmp, b2_tmp)

要把概率转换为决策，我们应用一个阈值：

In [ ]:
yhat = np.zeros_like(predictions)
for i in range(len(predictions)):
    if predictions[i] >= 0.5:
        yhat[i] = 1
    else:
        yhat[i] = 0
print(f"decisions = \n{yhat}")

这可以更简洁地完成：

In [ ]:
yhat = (predictions >= 0.5).astype(int)
print(f"decisions = \n{yhat}")

## 网络函数

这张图展示了整个网络的运行，结果与上一个实验中 Tensorflow 的结果完全相同。
左图是最后一层的原始输出，用蓝色阴影表示，叠加在用 X 和 O 表示的训练数据上。  
右图是经过决策阈值后网络的输出。这里的 X 和 O 对应网络做出的决策。  

In [ ]:
netf= lambda x : my_predict(norm_l(x),W1_tmp, b1_tmp, W2_tmp, b2_tmp)
plt_network(X,Y,netf)

## 恭喜！
你已经用 NumPy 构建了一个小型神经网络。  
希望这个实验揭示了构成神经网络中一层的那些相当简单而熟悉的函数。